### **Question 4: Sentence Probabilities Using Bigram Language Models**

This notebook implements and compares two N-gram language models:
1. **Unsmoothed Bigram Model** (Maximum Likelihood Estimation)
2. **Smoothed Bigram Model** (Laplace / Add-1 Smoothing)

The models are trained on the 3 sentences in `Data_3.txt` and evaluated on the target test sentence.

##### **1) Data Reading and Pre-processing**

In [ ]:
import os

# Locate Data_3.txt file path (handles both running from root or PART_A directory)
file_path = "DATASET/Data_3.txt" if os.path.exists("DATASET/Data_3.txt") else "PART_A/DATASET/Data_3.txt"

# Read all non-empty lines from the dataset
with open(file_path, "r") as file:
    raw_lines = [line.strip() for line in file if line.strip()]

# Separate training corpus sentences from the target test sentence
# Filter out header and decorative lines (e.g. 'Training Corpus', '~~~~~~~~~')
training_sentences = []
test_sentence = ""
is_training_section = True

for line in raw_lines:
    # When the section header for the test sentence appears, switch mode
    if "Calculate sentence probability" in line:
        is_training_section = False
        continue
    
    # Skip decorative separator lines or section titles
    if line.startswith("~") or "Training Corpus" in line:
        continue
    
    # Collect valid sentence lines containing sentence boundary pads
    if is_training_section:
        training_sentences.append(line)
    else:
        test_sentence = line

# Tokenize each sentence into words using whitespace splitting
# Note: Sentence pads <s> and </s> are preserved as distinct boundary tokens
train_tokens = [sentence.split() for sentence in training_sentences]
test_tokens = test_sentence.split()

# Display the pre-processed sentences and tokens
print("Training Corpus Sentences:")
for idx, sentence in enumerate(training_sentences, 1):
    print(f"  Sentence {idx}: {sentence}")

print("\nTest Sentence to Evaluate:")
print(f"  {test_sentence}")

print("\nTest Sentence Tokens:")
print(test_tokens)

##### **2) Frequency Counts: Unigrams, Bigrams, and Vocabulary (V)**

In [ ]:
from collections import Counter

# Initialise counters for unigrams (individual words) and bigrams (word pairs)
unigram_counts = Counter()
bigram_counts = Counter()

# Count unigrams and extract consecutive word pairs (bigrams) from the training sentences
for tokens in train_tokens:
    # Update unigram frequency counts
    unigram_counts.update(tokens)
    
    # Generate consecutive bigram pairs (w_{i-1}, w_i) and update bigram counts
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        bigram_counts[pair] += 1

# Vocabulary (V) consists of all unique tokens observed in the training corpus
vocab = sorted(list(unigram_counts.keys()))
V = len(vocab)

# Display vocabulary and frequency tables
print(f"Vocabulary Size (V): {V}")
print(f"Unique Tokens in Vocabulary: {vocab}\n")

print("Unigram Counts (used in conditional denominators):")
for word, count in sorted(unigram_counts.items()):
    print(f"  Count('{word}') = {count}")

print("\nBigram Counts (observed in training corpus):")
for pair, count in sorted(bigram_counts.items()):
    print(f"  Count({pair[0]}, {pair[1]}) = {count}")

##### **3) Unsmoothed Bigram Model**

The **unsmoothed bigram model** uses Maximum Likelihood Estimation (MLE):
$$
P(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1}, w_i)}{\text{Count}(w_{i-1})}
$$

The total sentence probability is the product of the conditional probabilities of all bigrams:
$$
P(\text{Sentence}) = \prod_{i=1}^{n} P(w_i \mid w_{i-1})
$$

In [ ]:
# Extract all bigrams from the test sentence
test_bigrams = [(test_tokens[i], test_tokens[i + 1]) for i in range(len(test_tokens) - 1)]

unsmoothed_prob = 1.0

print("Unsmoothed Bigram Probability Calculations:")
print(f"{'Bigram P(w2 | w1)':<32} {'Count(w1, w2)':<15} {'Count(w1)':<12} {'Formula':<12} {'Probability':<12}")
print("-" * 85)

for w1, w2 in test_bigrams:
    # Frequency of the pair (w1, w2) occurring consecutively in training corpus
    pair_count = bigram_counts[(w1, w2)]
    
    # Frequency of the history / conditioning word w1
    prev_count = unigram_counts[w1]
    
    # Calculate conditional probability using MLE formula: Count(w1, w2) / Count(w1)
    if prev_count == 0 or pair_count == 0:
        prob = 0.0
    else:
        prob = pair_count / prev_count
    
    # Multiply into the cumulative sentence probability
    unsmoothed_prob *= prob
    
    formula_str = f"{pair_count}/{prev_count}"
    print(f"P({w2:<10} | {w1:<10}) {pair_count:<15} {prev_count:<12} {formula_str:<12} {prob:.4f}")

print("-" * 85)
print(f"Total Unsmoothed Sentence Probability : {unsmoothed_prob:.6f}")
print(f"Exact Fraction Representation        : 1/27 ≈ {1/27:.6f}")

##### **4) Smoothed Bigram Model (Laplace / Add-1 Smoothing)**

To handle the zero-frequency problem where unseen pairs give zero probability, **Laplace (Add-1) smoothing** adds 1 to each bigram count and $V$ to the history count:
$$
P_{\text{smoothed}}(w_i \mid w_{i-1}) = \frac{\text{Count}(w_{i-1}, w_i) + 1}{\text{Count}(w_{i-1}) + V}
$$

Where:
- $+1$ in the numerator prevents zero probabilities for unseen bigrams.
- $+V$ in the denominator normalizes the probabilities so that $\sum_{w_i} P(w_i \mid w_{i-1}) = 1$.

In [ ]:
smoothed_prob = 1.0

print("Smoothed Bigram Probability Calculations (Laplace / Add-1):")
print(f"{'Bigram P(w2 | w1)':<32} {'Count+1':<12} {'Count(w1)+V':<15} {'Fraction':<12} {'Probability':<12}")
print("-" * 85)

for w1, w2 in test_bigrams:
    # Frequency of the pair (w1, w2)
    pair_count = bigram_counts[(w1, w2)]
    
    # Frequency of the conditioning word w1
    prev_count = unigram_counts[w1]
    
    # Laplace smoothing formula: (Count(w1, w2) + 1) / (Count(w1) + V)
    numerator = pair_count + 1
    denominator = prev_count + V
    prob = numerator / denominator
    
    # Multiply into cumulative sentence probability
    smoothed_prob *= prob
    
    fraction_str = f"{numerator}/{denominator}"
    print(f"P({w2:<10} | {w1:<10}) {f'{pair_count}+1={numerator}':<12} {f'{prev_count}+{V}={denominator}':<15} {fraction_str:<12} {prob:.6f}")

print("-" * 85)
print(f"Total Smoothed Sentence Probability (Scientific) : {smoothed_prob:.10e}")
print(f"Total Smoothed Sentence Probability (Decimal)    : {smoothed_prob:.12f}")
print(f"Exact Fraction Representation                    : 512 / 418161601 ≈ {512 / 418161601:.12f}")

##### **5) Comparison & Summary of Results**

In [ ]:
# Summary table comparing unsmoothed vs smoothed models
print("=" * 65)
print(f"{'Model':<30} {'Probability (Decimal)':<25} {'Fraction':<15}")
print("=" * 65)
print(f"{'Unsmoothed Bigram Model':<30} {unsmoothed_prob:<25.6f} {'1/27':<15}")
print(f"{'Smoothed Bigram Model (Add-1)':<30} {smoothed_prob:<25.10e} {'512/418161601':<15}")
print("=" * 65)

# Key observations:
# 1. Unsmoothed model assigns higher probability (0.037037) because all test bigrams happened to exist in the corpus.
#    However, if even ONE bigram was unseen, the unsmoothed probability would become 0.0.
# 2. Smoothed model produces a lower probability (1.2244e-06) because probability mass is redistributed
#    to all possible unseen word combinations across the vocabulary of size V = 10.